In [ ]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

In [ ]:
import os
os.listdir('/kaggle/input')

In [ ]:
# importing torch, torchvision for image-related tasks
import torch
import torchvision
from torchvision.datasets import MNIST

In [ ]:
# creating mnist dataset object 
dataset = MNIST(root='data/', download=True)

In [ ]:
len(dataset)

In [ ]:
# creating mnist dataset object for test datasets
test_dataset = MNIST(root='data/', train=False)
len(test_dataset)

In [ ]:
# importing matplotlib library for display
import matplotlib.pyplot as plt
%matplotlib inline

In [ ]:
# Displaying and image and its label
image, label = dataset[1]
plt.imshow(image, cmap='gray')
print("Label", label)

In [ ]:
# importing the transforms from torchvision for image preprocessing
import torchvision.transforms as transforms

In [ ]:
# using the transform to converts the numpy arrays to tensors
dataset = MNIST(
    root='data/',
    train=True,
    transform=transforms.ToTensor()
)

In [ ]:
img_tensor, label = dataset[0]
print(img_tensor.shape, label)

In [ ]:
# importing numpy for mathematical calculation
import numpy as np

# function for spliting the data
def split_indices(n, val_pct):
    n_val = int(val_pct*n)
    idxs = np.random.permutation(n)
    return idxs[n_val:], idxs[:n_val]

In [ ]:
# spliting the data into train, and validatin data
train_indices, val_indices = split_indices(len(dataset), val_pct=0.2)

In [ ]:
print(len(train_indices), len(val_indices))
print("Sample val indices: ", val_indices[:20])

In [ ]:
# importing random sampler and dataloader
from torch.utils.data.sampler import SubsetRandomSampler
from torch.utils.data.dataloader import DataLoader

In [ ]:
# Number of samples to load in each batch during training or validation
batch_size = 100

# Creates a random sampler that will draw samples only from the indices in 'train_indices'
train_sampler = SubsetRandomSampler(train_indices)

# DataLoader for the training set: loads batches of 'batch_size' using 'train_sampler' to select samples
train_loader = DataLoader(dataset,
                         batch_size,
                         sampler=train_sampler)

# Creates a random sampler that will draw samples only from the indices in 'val_indices'
val_sampler = SubsetRandomSampler(val_indices)

# DataLoader for the validation set: loads batches of 'batch_size' using 'val_sampler' to select samples
val_loader = DataLoader(dataset,
                       batch_size,
                       sampler=val_sampler)

In [ ]:
# Imports PyTorch's neural network module, which contains layers, loss functions, and model building blocks
import torch.nn as nn

# Each MNIST image is 28x28 pixels; flattening it gives 784 input features
input_size = 28*28
# There are 10 possible digit classes: 0 through 9
num_classes = 10

# Creates a single fully connected (linear) layer mapping 784 inputs to 10 outputs (logits for each digit)
model = nn.Linear(input_size, num_classes)

In [ ]:
# Printing the weight of the model and its shape
print(model.weight.shape)
model.weight

In [ ]:
# Printing the bias and its shape
print(model.bias.shape)
model.bias

In [ ]:
for images, labels in train_loader:
    print(labels)
    print(images.shape)
    outputs = model(images)
    break

In [ ]:
class MnistModel(nn.Module):  
    # Defines a custom neural network model for MNIST by subclassing nn.Module
    
    def __init__(self):  
        super().__init__()  
        # Calls the parent nn.Module constructor to properly initialize the model
        self.linear = nn.Linear(input_size, num_classes)  
        # Creates a single linear layer mapping 784 inputs to 10 outputs
    
    def forward(self, xb):  
        # Defines the forward pass
        xb = xb.reshape(-1, 784)  
        # Flattens the batch of images from (batch_size, 1, 28, 28) to (batch_size, 784)
        out = self.linear(xb)  
        # Passes the flattened images through the linear layer to get logits
        return out  
        # Returns the output logits (not yet passed through activation like softmax)

model = MnistModel()  
# Instantiates the model so it’s ready to be trained


In [ ]:
print(model.linear.weight.shape, model.linear.bias.shape)
list(model.parameters())

In [ ]:
for images, labels in train_loader:
    outputs = model(images)
    break

print('outputs.shape: ', outputs.shape)
print('Sample outputs :\n', outputs[:2].data)

In [ ]:
# Imports the functional API from torch.nn, which provides stateless functions
import torch.nn.functional as F

In [ ]:
probs = F.softmax(outputs, dim=1)  
# Applies the softmax function to 'outputs' along dimension 1 (the class dimension)
# This converts raw logits into probabilities that sum to 1 for each sample

print('Sample probabilities:\n', probs[:2].data)  
# Prints the predicted probabilities for the first 2 samples in the batch

print('Sum:', torch.sum(probs[0]).item())  
# Calculates and prints the sum of probabilities for the first sample
# This should be very close to 1.0 due to the softmax transformation

In [ ]:
max_probs, preds = torch.max(probs, dim=1)
print(preds)
print(max_probs)

In [ ]:
def accuracy(l1, l2):  
    # Computes the classification accuracy between two label tensors
    return torch.sum(l1 == l2).item() / len(l1)  
    # Counts how many predictions match the true labels, converts to a Python number, 
    # then divides by the total number of samples to get accuracy in [0,1]

In [ ]:
# Calculates the fraction of correct predictions in 'preds' compared to the true 'labels'
accuracy(preds, labels)

In [ ]:
# Sets the loss function to Cross Entropy Loss, which combines softmax and negative log-likelihood for classification
loss_fn = F.cross_entropy

In [ ]:
loss = loss_fn(outputs, labels)  
# Computes the cross-entropy loss between model predictions (outputs) and true labels

print(loss)  
# Displays the loss value for the current batch


**Optimization**

In [ ]:
learning_rate = 0.001  
# Step size for updating model parameters during training

optimizer = torch.optim.SGD(model.parameters(), lr=learning_rate)  
# Uses Stochastic Gradient Descent to update the model's parameters with the given learning rate


In [ ]:
# Defines a function to process one batch: computes loss, updates weights (if optimizer given), and calculates a metric.
def loss_batch(model, loss_func, xb, yb, opt=None, metric=None):

    # Forward pass: get predictions and compute loss
    preds = model(xb)
    loss = loss_func(preds, yb)

    # Backpropagation and weight update if training
    if opt is not None:
        loss.backward()
        opt.step()
        opt.zero_grad()

    # Compute metric (e.g., accuracy) if provided
    metric_result = None
    if metric is not None:
        metric_result = metric(preds, yb)

    # Return loss value, number of samples in batch, and metric result
    return loss.item(), len(xb), metric_result


In [ ]:
# Evaluates the model on a validation set without updating weights
def evaluate(model, loss_fn, valid_dl, metric=None):
    
    # Disable gradient tracking to save memory and speed up evaluation
    with torch.no_grad():

        # Process each batch in the validation DataLoader
        results = [loss_batch(model, loss_fn, xb, yb, metric=metric)
                   for xb, yb in valid_dl]

        # Unpack results into separate lists
        losses, nums, metrics = zip(*results)

        # Total number of samples
        total = np.sum(nums)

        # Weighted average loss across all batches
        avg_loss = np.sum(np.multiply(losses, nums)) / total

        # Weighted average metric (e.g., accuracy) if provided
        avg_metric = None
        if metric is not None:
            avg_metric = np.sum(np.multiply(metrics, nums)) / total

    # Return average loss, total samples, and average metric
    return avg_loss, total, avg_metric


In [ ]:
def accuracy(outputs, labels):
    _, preds = torch.max(outputs, dim=1)
    return torch.sum(preds == labels).item() / len(preds)

In [ ]:
val_loss, total, val_acc = evaluate(model, loss_fn, val_loader, metric=accuracy)
print('Loss: {:.4f}, Accuracy: {:.4f}'.format(val_loss, val_acc))

In [ ]:
# Trains the model for a given number of epochs and evaluates it after each epoch
def fit(epochs, model, loss_fn, opt, train_dl, valid_dl, metric=None):
    
    # Loop over epochs
    for epoch in range(epochs):

        # Loop over training batches and update weights
        for xb, yb in train_dl:
            loss, _, _ = loss_batch(model, loss_fn, xb, yb, opt)

        # Evaluate on the validation set
        result = evaluate(model, loss_fn, valid_dl, metric)
        val_loss, total, val_metric = result

        # Print loss and metric for this epoch
        if metric is None:
            print('Epoch [{}/{}], Loss: {:.4f}'
                  .format(epoch+1, epochs, val_loss))
        else:
            print('Epoch [{}/{}], Loss: {:.4f}, {}: {:.4f}'
                  .format(epoch+1, epochs, val_loss, metric.__name__, val_metric))

In [ ]:
model = MnistModel()
optimizer = torch.optim.SGD(model.parameters(), lr=learning_rate)

In [ ]:
fit(5, model, F.cross_entropy, optimizer, train_loader, val_loader, accuracy)

In [ ]:
fit(5, model, F.cross_entropy, optimizer, train_loader, val_loader, accuracy)

In [ ]:
fit(5, model, F.cross_entropy, optimizer, train_loader, val_loader, accuracy)

In [ ]:
fit(20, model, F.cross_entropy, optimizer, train_loader, val_loader, accuracy)

In [ ]:
accuracies = [0.8008, 0.8324, 0.8484, 0.8706]

plt.plot(accuracies, '-x')
plt.xlabel('epoch')
plt.ylabel('accuracy')
plt.title('Accuracy vs. No. of epochs')

In [ ]:
test_dataset = MNIST(root='data/',
                    train=False,
                    transform=transforms.ToTensor())

In [ ]:
def predict_image(img, model):
    xb = img.unsqueeze(0)
    yb = model(xb)
    _, preds = torch.max(yb, dim=1)
    return preds[0].item()

In [ ]:
img, label = test_dataset[0]
plt.imshow(img[0], cmap='gray')
print("Label:", label, ', Predicted: ', predict_image(img, model))

In [ ]:
img, label = test_dataset[10]
plt.imshow(img[0], cmap='gray')
print("Label:", label, ', Predicted: ', predict_image(img, model))

In [ ]:
img, label = test_dataset[196]
plt.imshow(img[0], cmap='gray')
print("Label:", label, ', Predicted: ', predict_image(img, model))

In [ ]:
img, label = test_dataset[1839]
plt.imshow(img[0], cmap='gray')
print("Label:", label, ', Predicted: ', predict_image(img, model))

In [ ]:
test_loader = DataLoader(test_dataset, batch_size=200)

test_loss, total, test_acc = evaluate(model, loss_fn, test_loader, metric=accuracy)
print('Loss: {:.4f}, Accuracy: {:.4f}'.format(test_loss, test_acc))

torch.save(model.state.dict(), 'mnist)

In [ ]:
# Saving the model
torch.save(model.state_dict(), 'mnist-logistic.pth')

In [ ]:
model.state_dict()

In [ ]:
# Loading and testing the model
model2 = MnistModel()
model2.load_state_dict(torch.load('mnist-logistic.pth'))
model2.state_dict()

In [ ]:
test_loss, total, test_acc = evaluate(model2, loss_fn, test_loader, metric=accuracy)
print('Loss: {:.4f}, Accuracy: {:.4f}'.format(test_loss, test_acc))